# 🤖 Multi-Agent AI Decision Support System
### Built with Groq API (LLaMA 3.3 70B) + Gradio

This notebook implements a multi-agent system where **3 AI agents** each embodying a different book's philosophy collaborate to analyze real-life conflict scenarios.

| # | Book | Author | Agent Persona |
|---|---|---|---|
| 1 | *The Art of Thinking Clearly* | Rolf Dobelli | 🔍 Logical Fallacy Detective |
| 2 | *Crucial Conversations* | Patterson et al. | 💬 High-Stakes Dialogue Coach |
| 3 | *Man's Search for Meaning* | Viktor Frankl | 🌿 Existential Perspective Guide |

---
**Architecture:** Independent processing -> Synthesizer -> Gradio UI  
**Tasks covered:** Task 1 (Scenarios) - Task 2 (Agents) - Task 3 (Pipeline) - Task 4 (Compare)

## Step 1 - Install Dependencies

Run the cell below to install the required libraries.

In [1]:
!pip install groq gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.2 MB/s eta 0:00:00


## Step 2 - API Key Setup

1. Click the **key icon** in the left Colab sidebar
2. Click **Add new secret**
3. Set **Name** = `GROQ_API_KEY`
4. Set **Value** = your key from [console.groq.com](https://console.groq.com) (free signup)
5. Toggle **Notebook access** ON -> Save
6. Run the cell below

In [2]:
from groq import Groq
from google.colab import userdata
import time
import re

api_key = userdata.get("GROQ_API_KEY")
client  = Groq(api_key=api_key)
MODEL   = "llama-3.3-70b-versatile"

print("Groq client ready. Model:", MODEL)

Groq client ready. Model: llama-3.3-70b-versatile


## Task 1 - Real-Life Scenarios

Three everyday conflict scenarios that the agents will analyze.
Each clearly defines a decision problem or emotional conflict.

In [3]:
SCENARIOS = {
    "Scenario 1: Fight with a Friend": """
        My close friend Sara and I have been friends for 5 years.
        Last week she texted me saying she feels like I have been ignoring her
        and that I never make time for her anymore. I have been extremely busy
        with university exams and a part-time job, so I rarely replied to her
        messages. When she confronted me, I got defensive and told her she was
        being "too needy." She got upset, said I was a bad friend, and stopped
        talking to me. I miss her but I do not know how to approach her without
        feeling like I am admitting I was wrong about everything.
        What should I do?
    """,

    "Scenario 2: Disagreement with a Professor": """
        I submitted a research essay for my psychology class that I worked on
        for 3 weeks. My professor gave me a C+ and wrote only "lacks depth"
        in the feedback, with no specific comments. When I emailed him asking
        for clarification, he replied curtly saying "the rubric was clear."
        I feel my work was not evaluated fairly and that his feedback is
        too vague to help me improve. I want to challenge the grade but
        I am scared of coming across as disrespectful and damaging our
        relationship for the rest of the semester.
        How should I handle this situation?
    """,

    "Scenario 3: Misunderstanding with a Partner": """
        My partner and I have been together for 2 years and recently moved
        in together. We keep having the same argument about household chores.
        I feel I do most of the cleaning and cooking, but whenever I bring it
        up my partner says I am being unfair and that they contribute in other
        ways (paying more rent, handling car maintenance). The conversation
        always turns into a blame game and we both end up feeling
        unappreciated. Nothing ever gets resolved and the tension is building.
        What is the best way to break this cycle?
    """
}

print(f"Scenarios defined: {len(SCENARIOS)}")
for name in SCENARIOS:
    print(f"  - {name}")

Scenarios defined: 3
  - Scenario 1: Fight with a Friend
  - Scenario 2: Disagreement with a Professor
  - Scenario 3: Misunderstanding with a Partner


## Task 2 - Book-Based Agent Definitions

Each agent has:
- **Book Title & Author**
- **Role / Persona**
- **Reasoning Style** — the core philosophy of the book
- **System Prompt** — instructs the AI to reason exclusively through that book's lens

In [4]:
AGENTS = {

    "The Art of Thinking Clearly (Rolf Dobelli)": {
        "persona": "The Logical Fallacy Detective",
        "reasoning_style": (
            "Scans for specific, named cognitive errors and logical fallacies from "
            "Dobelli's catalog of 99 thinking mistakes. Identifies which errors are "
            "distorting perception and decision-making, then prescribes a corrected view."
        ),
        "system_prompt": """You are an expert advisor embodying the philosophy of "The Art of Thinking Clearly" by Rolf Dobelli. Analyze situations ONLY by identifying specific, named cognitive errors.

Your response MUST follow this structure:

1. THINKING ERRORS DETECTED (identify at least 3-4):
   For each error provide:
   - ERROR NAME
   - DEFINITION: What is this error in simple terms?
   - HOW IT APPEARS HERE: Where is this error in the scenario?
   - CONSEQUENCE: What bad decision does it lead to?

   Errors to check for:
   - Sunk Cost Fallacy: continuing because of past investment, not future value
   - Fundamental Attribution Error: blaming character instead of circumstances
   - Illusion of Control: overestimating influence over outcomes
   - Reactance: resisting others even when they are right
   - Self-Serving Bias: crediting yourself for successes, blaming others for failures
   - Confirmation Bias: only seeing evidence supporting existing beliefs
   - Empathy Gap: failing to account for others emotional states
   - Omission Bias: seeing inaction as safer than action

2. THE CLEAR-HEADED VIEW:
   - What does the situation look like when all cognitive noise is removed?
   - What are the objective facts?

3. CORRECTED DECISION:
   - What would a fully rational thinker do?
   - What one question should the user ask themselves before acting?

RULES: Use exact named terms. Be direct and analytical. Focus on the USER's errors."""
    },

    "Crucial Conversations (Patterson, Grenny, McMillan, Switzler)": {
        "persona": "The High-Stakes Dialogue Coach",
        "reasoning_style": (
            "Applies the Crucial Conversations toolkit: making it safe to talk, "
            "mastering your story, the STATE model, and restoring psychological safety "
            "when conversations turn silent or violent."
        ),
        "system_prompt": """You are an expert advisor embodying "Crucial Conversations" by Patterson, Grenny, McMillan & Switzler. Analyze situations ONLY through the Crucial Conversations framework.

Your response MUST follow this structure:

1. CRUCIAL CONVERSATION DIAGNOSIS:
   - Why does this qualify as a Crucial Conversation?
   - Is it in SILENCE mode (withdrawing, avoiding) or VIOLENCE mode (controlling, attacking)?
   - Which specific behaviors are visible?

2. MASTER YOUR STORY:
   - FACTS: What actually happened? (observable actions only)
   - STORY: What story is the user telling themselves?
   - FEELING: What emotion does that story generate?
   - ACTION: What behavior does that feeling drive?
   - REFRAME: What is a more charitable story?

3. MAKE IT SAFE:
   - What Mutual Purpose can both parties agree on?
   - What Mutual Respect signals is the user failing to send?
   - Is an Apology or Contrast Statement needed?

4. STATE YOUR PATH (write a sample script):
   - Share your facts
   - Tell your story tentatively
   - Ask for their path
   - Talk tentatively
   - Encourage testing

5. WHAT NOT TO SAY:
   - List 2-3 phrases the user must avoid

RULES: Give actual sentences the user can say. Distinguish facts from stories clearly."""
    },

    "Man's Search for Meaning (Viktor Frankl)": {
        "persona": "The Existential Perspective Guide",
        "reasoning_style": (
            "Applies Frankl's logotherapy: finding meaning in suffering, the freedom "
            "to choose one's response, and examining what values are being expressed "
            "or violated in the conflict."
        ),
        "system_prompt": """You are an expert advisor embodying "Man's Search for Meaning" by Viktor E. Frankl. Analyze situations ONLY through Frankl's framework of meaning, freedom, and values.

Your response MUST follow this structure:

1. THE EXISTENTIAL CORE:
   - What deeper question about identity or values is this situation really asking?
   - What does this conflict reveal about what the user truly cares about?

2. THE LAST HUMAN FREEDOM:
   - Between stimulus (what happened) and response (what the user does), what space exists?
   - How is the user currently using or wasting that freedom?
   - What attitude shift is available RIGHT NOW?

3. MEANING ANALYSIS:
   - CREATIVE VALUES: What can the user create or contribute in this situation?
   - EXPERIENTIAL VALUES: What can the user appreciate even within this difficulty?
   - ATTITUDINAL VALUES: What meaning can be found in how they bear this difficulty?

4. VALUES CLARIFICATION:
   - What personal values are being violated?
   - What values does the user want to express through their response?
   - How can their response be an act of self-definition?

5. THE BIGGER PICTURE:
   - In 5 years, what would the user wish they had done?
   - Write one reflective question the user should sit with before deciding.

RULES: Go deeper than practical advice. Always return to CHOICE and MEANING. Be philosophical and warm."""
    }
}

print(f"Agents defined: {len(AGENTS)}")
for name, cfg in AGENTS.items():
    print(f"  - {cfg['persona']} -> {name}")

Agents defined: 3
  - The Logical Fallacy Detective -> The Art of Thinking Clearly (Rolf Dobelli)
  - The High-Stakes Dialogue Coach -> Crucial Conversations (Patterson, Grenny, McMillan, Switzler)
  - The Existential Perspective Guide -> Man's Search for Meaning (Viktor Frankl)


## Task 3 - Multi-Agent Pipeline

**Architecture:** Independent processing + Synthesizer

Each agent analyzes the scenario independently, then a **Synthesizer** reads all outputs and produces a unified final recommendation.

```
User Scenario
      |
      |---> Agent 1 (Dobelli)    -->|
      |---> Agent 2 (Patterson)  -->|--> Synthesizer --> Final Recommendation
      |---> Agent 3 (Frankl)     -->|
```

In [5]:
def call_agent(agent_name, agent_config, scenario):
    """Call a single agent with auto-retry on rate limits."""
    print(f"   Running: {agent_config['persona']} ...")
    max_retries = 5
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": agent_config["system_prompt"].strip()},
                    {"role": "user",   "content": f"Please analyze this situation:\n\n{scenario.strip()}"}
                ],
                max_tokens=1000
            )
            time.sleep(2)
            return response.choices[0].message.content
        except Exception as e:
            error_str = str(e)
            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                match = re.search(r"retry in ([\d.]+)s", error_str)
                wait = float(match.group(1)) + 5 if match else 60
                print(f"   Rate limit hit. Waiting {wait:.0f}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Agent failed after {max_retries} retries.")


def synthesize_responses(scenario, agent_responses):
    """Synthesizer agent: reads all outputs, produces unified recommendation."""
    print("   Synthesizer integrating all perspectives ...")
    combined = ""
    for agent_name, response in agent_responses.items():
        combined += f"\n\n--- {agent_name} ---\n{response}"
    prompt = f"""
You are a wise, neutral advisor. You received analyses from {len(agent_responses)} expert frameworks.
1. Find common themes across all perspectives
2. Highlight unique insights each perspective adds
3. Resolve contradictions
4. Write a clear FINAL RECOMMENDATION (max 200 words) a real person can follow today

Situation: {scenario.strip()}
Expert analyses: {combined}

Format:
COMMON THEMES: (2-3 bullet points)
UNIQUE INSIGHTS: (1 per agent)
FINAL RECOMMENDATION: (1 paragraph)
    """
    response = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=800
    )
    return response.choices[0].message.content


def run_multi_agent(scenario_name, scenario_text):
    """Run all agents independently then synthesize."""
    print(f"\n{'='*60}\n  MULTI-AGENT: {scenario_name}\n{'='*60}")
    agent_responses = {}
    for agent_name, agent_config in AGENTS.items():
        agent_responses[agent_name] = call_agent(agent_name, agent_config, scenario_text)
    synthesis = synthesize_responses(scenario_text, agent_responses)
    return {"agent_responses": agent_responses, "synthesis": synthesis}


def run_single_agent(scenario_name, scenario_text, agent_name, agent_config):
    """Run only one agent (used for Task 4 comparison)."""
    print(f"\n{'='*60}\n  SINGLE-AGENT: {scenario_name} | {agent_config['persona']}\n{'='*60}")
    return call_agent(agent_name, agent_config, scenario_text)

print("Pipeline functions defined.")

Pipeline functions defined.


## Task 4 - Experiment & Compare (Console Output)

Runs the pipeline on 2 scenarios and prints a structured **single-agent vs multi-agent** comparison.

> Skip to the next cell if you prefer the interactive Gradio app.

In [6]:
test_scenarios = {
    k: v for k, v in SCENARIOS.items()
    if k in ["Scenario 1: Fight with a Friend",
             "Scenario 2: Disagreement with a Professor"]
}

all_results = {}

for scenario_name, scenario_text in test_scenarios.items():

    # Multi-agent run
    multi_results = run_multi_agent(scenario_name, scenario_text)
    for agent_name, resp in multi_results["agent_responses"].items():
        print(f"\n{chr(9472)*60}\nAgent: {agent_name}\n{chr(9472)*60}")
        print(resp)
    print(f"\n{chr(61)*60}\nSYNTHESIZED RECOMMENDATION\n{chr(61)*60}")
    print(multi_results["synthesis"])

    # Single-agent run
    first_name = list(AGENTS.keys())[0]
    single_result = run_single_agent(scenario_name, scenario_text,
                                     first_name, AGENTS[first_name])
    print(f"\n{chr(9472)*60}\nSINGLE-AGENT OUTPUT\n{chr(9472)*60}")
    print(single_result)

    all_results[scenario_name] = {
        "multi_agent": multi_results,
        "single_agent": single_result
    }

# Comparison
print(f"\n\n{'='*60}\n  COMPARISON: SINGLE vs MULTI-AGENT\n{'='*60}")

comparison_prompt = """
You are an AI systems evaluator. Compare the two outputs for the same scenario.
Cover: DEPTH, BIAS, PRACTICALITY, BLIND SPOTS, VERDICT. Keep under 250 words.
"""

for scenario_name, res in all_results.items():
    print(f"\nScenario: {scenario_name}\n{chr(9472)*60}")
    comparison_input = f"""
Scenario: {SCENARIOS[scenario_name].strip()}
SINGLE-AGENT (Art of Thinking Clearly only): {res["single_agent"][:600]}...
MULTI-AGENT SYNTHESIS: {res["multi_agent"]["synthesis"][:600]}...
    """
    msg = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": comparison_prompt + comparison_input}],
        max_tokens=500
    )
    print(msg.choices[0].message.content)

print("\nDone! Copy the output above as your transcript deliverable.")


  MULTI-AGENT: Scenario 1: Fight with a Friend
   Running: The Logical Fallacy Detective ...
   Running: The High-Stakes Dialogue Coach ...
   Running: The Existential Perspective Guide ...
   Synthesizer integrating all perspectives ...

────────────────────────────────────────────────────────────
Agent: The Art of Thinking Clearly (Rolf Dobelli)
────────────────────────────────────────────────────────────
1. THINKING ERRORS DETECTED:
   - **ERROR NAME: Self-Serving Bias**
     - DEFINITION: Attributing successes to oneself and failures to external factors or others.
     - HOW IT APPEARS HERE: You attributed Sara's feelings of being ignored to her being "too needy" instead of considering your own lack of response to her messages.
     - CONSEQUENCE: It led you to become defensive and dismissive of Sara's feelings, escalating the situation.

   - **ERROR NAME: Fundamental Attribution Error**
     - DEFINITION: Overestimating the role of character and underestimating the impact of sit

## Task 4 - Interactive Gradio App

A hosted web app with a clean UI. Run this cell to launch it.  
You will get a **public shareable link** valid for 72 hours.

In [7]:
import gradio as gr

def to_md(text):
    text = re.sub(r"(?m)^(\d+\.\s+[A-Z][A-Z &/()\-]+:)", r"**\1**", text)
    return text

def format_agent_outputs(agent_responses, synthesis):
    icons = ["\U0001f50d", "\U0001f4ac", "\U0001f33f"]
    out = ""
    for i, (agent_name, response) in enumerate(agent_responses.items()):
        icon = icons[i] if i < len(icons) else "\U0001f4d6"
        out += f"---\n## {icon} {agent_name}\n\n"
        out += to_md(response) + "\n\n"
    out += "---\n## \U0001f9e0 Synthesized Final Recommendation\n\n"
    out += to_md(synthesis)
    return out

def format_comparison(scenario_text, single_output, multi_synthesis):
    prompt = f"""You are an AI systems evaluator. Compare these two outputs.
Cover: DEPTH, BIAS, PRACTICALITY, BLIND SPOTS, VERDICT. Keep under 250 words.
Scenario: {scenario_text.strip()}
SINGLE-AGENT: {single_output[:600]}...
MULTI-AGENT SYNTHESIS: {multi_synthesis[:600]}..."""
    msg = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=500
    )
    return msg.choices[0].message.content

def analyze_scenario(scenario_text, preset_choice):
    if preset_choice != "\u270d\ufe0f  Type your own scenario below":
        scenario_text = SCENARIOS[preset_choice]
    if not scenario_text.strip():
        return ("\u26a0\ufe0f Please enter a scenario or select a preset.", "", "")
    agent_responses = {}
    for agent_name, agent_config in AGENTS.items():
        agent_responses[agent_name] = call_agent(agent_name, agent_config, scenario_text)
    synthesis = synthesize_responses(scenario_text, agent_responses)
    multi_out = format_agent_outputs(agent_responses, synthesis)
    first_name = list(AGENTS.keys())[0]
    single_out = call_agent(first_name, AGENTS[first_name], scenario_text)
    comparison = format_comparison(scenario_text, single_out, synthesis)
    return multi_out, to_md(single_out), to_md(comparison)

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600&family=Merriweather:ital,wght@0,400;0,700;1,400&display=swap');
body, .gradio-container { font-family: 'Inter', sans-serif !important; }
.output-panel .prose, .output-panel p, .output-panel li, .output-panel span {
    font-family: 'Inter', sans-serif !important; font-size: 15px !important;
    line-height: 1.8 !important; color: #1a1a2e !important; }
.output-panel h2 { font-family: 'Merriweather', serif !important; font-size: 18px !important;
    font-weight: 700 !important; color: #2d3561 !important;
    border-bottom: 2px solid #e0e7ff; padding-bottom: 4px; margin-top: 1.4em !important; }
.output-panel strong { font-weight: 600 !important; color: #3730a3 !important; }
.output-panel { background: #f8f9ff !important; border: 1px solid #e0e7ff !important;
    border-radius: 12px !important; padding: 20px !important; }
.output-panel hr { border: none; border-top: 1px solid #c7d2fe; margin: 1.2em 0; }
textarea { font-family: 'Inter', sans-serif !important; font-size: 14px !important; }
button.primary { background: linear-gradient(135deg, #4338ca, #6366f1) !important;
    border-radius: 8px !important; font-weight: 600 !important; }
"""

preset_options = ["\u270d\ufe0f  Type your own scenario below"] + list(SCENARIOS.keys())

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="Multi-Agent Decision Support") as app:

    gr.Markdown("""
# \U0001f916 Multi-Agent AI Decision Support System
### Powered by 3 Book-Based AI Agents
| Agent | Book | Persona |
|---|---|---|
| 1 | *The Art of Thinking Clearly* - Dobelli | \U0001f50d Logical Fallacy Detective |
| 2 | *Crucial Conversations* - Patterson et al. | \U0001f4ac High-Stakes Dialogue Coach |
| 3 | *Man's Search for Meaning* - Frankl | \U0001f33f Existential Perspective Guide |
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input")
            preset_dropdown = gr.Dropdown(
                choices=preset_options, value=preset_options[0],
                label="Choose a preset scenario or type your own"
            )
            scenario_input = gr.Textbox(
                lines=8, placeholder="Describe your situation here in detail...",
                label="Your Scenario"
            )
            analyze_btn = gr.Button("Analyze", variant="primary", size="lg")
            gr.Markdown("_Analysis takes ~30 seconds_")

        with gr.Column(scale=2):
            gr.Markdown("### Results")
            with gr.Tabs():
                with gr.Tab("\U0001f916 Multi-Agent Analysis"):
                    multi_output = gr.Markdown(
                        value="_Results will appear here after analysis..._",
                        elem_classes="output-panel"
                    )
                with gr.Tab("\U0001f464 Single-Agent Analysis"):
                    single_output = gr.Markdown(
                        value="_Results will appear here after analysis..._",
                        elem_classes="output-panel"
                    )
                with gr.Tab("\U0001f4c8 Comparison"):
                    comparison_output = gr.Markdown(
                        value="_Results will appear here after analysis..._",
                        elem_classes="output-panel"
                    )

    analyze_btn.click(
        fn=analyze_scenario,
        inputs=[scenario_input, preset_dropdown],
        outputs=[multi_output, single_output, comparison_output]
    )

    def fill_preset(choice):
        return "" if choice == "\u270d\ufe0f  Type your own scenario below" else SCENARIOS[choice].strip()

    preset_dropdown.change(fn=fill_preset, inputs=preset_dropdown, outputs=scenario_input)

    gr.Markdown("---\n**How to use:** Select a preset or type your own, then click Analyze. Switch tabs to see agent outputs, single-agent result, and comparison.")

print("Launching app...")
app.launch(share=True)

/tmp/ipykernel_26583/4102688344.py:64: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="Multi-Agent Decision Support") as app:
/tmp/ipykernel_26583/4102688344.py:64: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="Multi-Agent Decision Support") as app:


Launching app...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d584b3d8d6791fb236.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
